# Unsloth GRPO Training for Hierarchical Reasoning

This notebook uses Unsloth's optimized kernels with TRL's GRPOTrainer for stable training.

**Key features:**
- ~50% less VRAM usage than standard transformers
- vLLM fast inference for generation
- HICRA-inspired reward functions for reasoning

In [ ]:
STRATEGIC_GRAMS = [
  "a better way",
  "actually, for real",
  "ah, i see",
  "ah, yes! i assumed",
  "and already discarded",
  "as above, we got",
  "as i said, it's the",
  "as is standard",
  "as long as",
  "as possible, but without",
  "be the case",
  "because from above",
  "boxed at the end",
  "but better to keep",
  "but for the guarantee, we",
  "but i need the maximum",
  "but in my earlier",
  "but in the original",
  "but it's poorly worded",
  "but perhaps it's in contrast",
  "but since there's no context",
  "but that probably won't",
  "but that's fine",
  "but usually it's specified",
  "but we have disagreements.",
  "combining these, i get",
  "confirm with small",
  "consider the behavior",
  "constraints are there",
  "defined in terms of",
  "discrepancy is due to",
  "doesn't make sense",
  "don't think it's necessary",
  "earlier we had two choices",
  "earlier when we solved",
  "eventually we might get to",
  "find a pattern",
  "first, compute the",
  "for this specific form",
  "from above, it's not",
  "good. the other option",
  "i can choose",
  "i can split this",
  "i have two options",
  "i recall that",
  "i see that",
  "i see, i forgot",
  "i think i double-counted",
  "i think i'm good",
  "i think i'm overthinking",
  "i think it's clear that",
  "i think it's safer",
  "i think it's solid",
  "i think it's something",
  "i think that's all",
  "i think the most",
  "i'll assume that",
  "i'll write the solution as",
  "if i set",
  "in fact, comparing, we",
  "in my initial setup",
  "is a key point",
  "is a known",
  "is close, but",
  "is computed correctly.",
  "is there a relationship",
  "is what we want for",
  "it is a bit circular",
  "it should hold",
  "it's an approximation",
  "later if needed",
  "let's call this",
  "likely it's not",
  "likely the order is",
  "likely the values",
  "look at the list",
  "look for similar problems",
  "made a mistake",
  "manually to verify",
  "maybe it's a system",
  "maybe it's a trick",
  "maybe just state",
  "meaning it cannot serve",
  "meaning we start",
  "mistake. let me go back.",
  "more importantly, to",
  "most straightforward interpretation is",
  "need to be careful",
  "no new information",
  "nothing else. so no.",
  "now for a random",
  "now the main parts",
  "now, another point",
  "now, are there others?",
  "now, are these both",
  "now, check the",
  "now, for other values",
  "now, from the solution",
  "now, in general",
  "now, is that the",
  "now, let me adjust the",
  "now, let me perform",
  "now, let me write",
  "now, let's go through",
  "now, next, this",
  "now, similarly for",
  "now, so far, we have",
  "now, there's a condition",
  "now, this makes sense.",
  "now, to combine",
  "now, we need another",
  "now, what if",
  "oh! that's better",
  "or perhaps it's",
  "or perhaps it's part of",
  "or something close to it",
  "other solution? for example",
  "perhaps it's the other",
  "probably related to",
  "probably, since the",
  "read it carefully",
  "refer to the underlying",
  "relate this to the",
  "safe, i should include",
  "say, for example, let",
  "seemed to work, but when",
  "seems large, but",
  "should be extended",
  "similarly for other",
  "similarly, from symmetry or",
  "similarly, it also requires",
  "since it's not required",
  "since we're interested in",
  "so all good",
  "so indeed, at least",
  "so we need",
  "so we only care",
  "so yes, confirmed",
  "so yes, it's exactly",
  "so, first, we",
  "so, i'll box that",
  "solutions to this might be",
  "split this into two",
  "standard notation: let\u2019s say",
  "still same issue",
  "suggests that maybe",
  "suppose i take",
  "suppose we define",
  "that i'm missing",
  "that's a good point",
  "that's a huge simplification",
  "that's a nice relation",
  "that's a nice simplification!",
  "that's a special case.",
  "that's one type",
  "that's the innermost part",
  "that's what i thought",
  "the general method to",
  "the maximum? perhaps",
  "the only possibilities",
  "the way it's phrased",
  "this is a classic",
  "this is trickier",
  "this is under the",
  "this looks tricky",
  "this would be violated.",
  "to be precise. sometimes",
  "to confirm, let's think",
  "to have agreement at least",
  "to my initial plan",
  "to simplify, i can",
  "try assuming that first",
  "viewed from above i think",
  "we already handled",
  "we are back to",
  "we are treating",
  "we assumed that",
  "we have two cases",
  "we need to ensure",
  "we need to satisfy",
  "we saw earlier",
  "we saw it is impossible",
  "what about when",
  "what's being asked",
  "where did i go",
  "which may not cover",
  "with this method",
  "yes, that's correct"
]

In [3]:
# Cell 1: Environment Setup.
import os
os.environ["fix_mistral_regex"] = "True"
# os.environ["OMP_NUM_THREADS"] = "1"
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # Extra 30% context lengths

# Install dependencies (run this if not already installed)
# !pip install unsloth vllm
# !pip install transformers==4.56.2
# !pip install --no-deps trl==0.22.2

In [4]:
# Set remote HF_TOKEN from local .env
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')

# ssh -i ~/.ssh/id_ed25519 dataimaginations-heirarchical-reasoning@ssh.hf.space "echo 'export HF_TOKEN={hf_token}' >> ~/.bashrc"
print("✅ Token set! Restart remote shell to activate.")

✅ Token set! Restart remote shell to activate.


In [5]:
# Cell 2: HuggingFace Login
import os
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print("✅ Logged in with HF_TOKEN")
else:
    login()
    print("✅ Logged in interactively")

✅ Logged in with HF_TOKEN


In [1]:
from unsloth import FastLanguageModel
import torch

# Configuration
max_seq_length = 2048
lora_rank = 128      # <--- Bump this to 128 (Since you already did 64!)
lora_alpha = 128     # <--- Generally keep Alpha = Rank for Unsloth

print(f"⏳ Loading model with Rank {lora_rank}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/phi-4-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False,
)

print("🔗 Attaching LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,        # <--- FIX: Use the variable, don"t hardcode!
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_alpha, # Set this to match the rank
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print(f"✅ Model loaded with Rank {lora_rank}!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Loading model with Rank 128...
==((====))==  Unsloth 2025.12.9: Fast Llama patching. Transformers: 4.57.3. vLLM: 0.13.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.484 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔗 Attaching LoRA adapters...


Unsloth 2025.12.9 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


✅ Model loaded with Rank 128!


Tuning Tips:
- If you"re getting OOM: Lower NEMOTRON_SAMPLE_SIZE to 1000-2000
- If generations are too long: Lower MAX_ANSWER_TOKENS to 400
- If you want more data: Increase NEMOTRON_SAMPLE_SIZE to 5000+
The filtering keeps ~60-70% of examples typically, so 3000 samples → ~2000 usable examples mixed with your 729 HICRA examples.



In [6]:
# Cell 4: Load and Combine Datasets
from datasets import load_dataset, Dataset
import json

# === Configuration ===
MAX_PROMPT_TOKENS = 400    # Filter out prompts longer than this
MAX_ANSWER_TOKENS = 600    # Filter out answers longer than this  
NEMOTRON_SAMPLE_SIZE = 3000  # How many Nemotron examples to use

# System prompt for reasoning format
SYSTEM_PROMPT = """
You are aadvanced mathematical, logical reasoning assistant. Think through problems step by step.
The calculation is as important as the answer.
Respond in the following format:
<think>
...
</think>
<answer>
...
</answer>
"""

def format_prompt(example):
    """Format dataset for GRPO training with chat template."""
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT.strip()},
            {"role": "user", "content": example['prompt']}
        ],
        'answer': str(example['answer'])
    }

def format_nemotron(example):
    """Convert Nemotron format to our format."""
    messages = example.get('messages', [])
    
    # Extract user prompt and assistant answer
    user_content = ""
    assistant_content = ""
    
    for msg in messages:
        if msg['role'] == 'user':
            user_content = msg['content']
        elif msg['role'] == 'assistant':
            assistant_content = msg['content']
    
    # Get expected answer (fallback to assistant content if not available)
    expected = example.get('expected_answer', '')
    if not expected:
        # Try to extract from assistant's <answer> tags if present
        if '<answer>' in assistant_content and '</answer>' in assistant_content:
            expected = assistant_content.split('<answer>')[-1].split('</answer>')[0].strip()
        else:
            expected = assistant_content[-200:] if len(assistant_content) > 200 else assistant_content
    
    return {
        'prompt': user_content,
        'answer': str(expected)
    }

def estimate_tokens(text):
    """Rough token estimate (1 token ≈ 4 chars for English)."""
    return len(str(text)) // 4

def filter_by_length(example):
    """Filter out examples that are too long."""
    prompt_tokens = estimate_tokens(example['prompt'])
    answer_tokens = estimate_tokens(example['answer'])
    return prompt_tokens <= MAX_PROMPT_TOKENS and answer_tokens <= MAX_ANSWER_TOKENS

# === 1. Load Your HICRA Synthetic Data ===
print("📂 Loading HICRA dataset...")
my_dataset = load_dataset(
    "json", 
    data_files="reasoning_dataset_v2_train.json", 
    split="train"
)
print(f"   ✅ Loaded {len(my_dataset)} HICRA examples")

# === 2. Load Nemotron Math Data (Streaming) ===
print(f"🌊 Streaming {NEMOTRON_SAMPLE_SIZE} Nemotron math examples...")
try:
    nemotron_stream = load_dataset(
        "nvidia/Nemotron-Post-Training-Dataset-v1", 
        split="math", 
        streaming=True
    )
    
    # Take a sample and convert to list
    nemotron_list = []
    for i, example in enumerate(nemotron_stream):
        if i >= NEMOTRON_SAMPLE_SIZE:
            break
        formatted = format_nemotron(example)
        # Only keep if it's not too long
        if filter_by_length(formatted):
            nemotron_list.append(formatted)
        
        if (i + 1) % 500 == 0:
            print(f"   Processed {i + 1} examples, kept {len(nemotron_list)}...")
    
    nemotron_dataset = Dataset.from_list(nemotron_list)
    print(f"   ✅ Loaded {len(nemotron_dataset)} Nemotron examples (after length filter)")
    
except Exception as e:
    print(f"   ⚠️ Could not load Nemotron: {e}")
    print("   Continuing with HICRA data only...")
    nemotron_dataset = None

# === 3. Combine Datasets ===
print("🔀 Combining datasets...")

# Filter HICRA by length too
my_dataset_filtered = my_dataset.filter(filter_by_length)
print(f"   HICRA after filter: {len(my_dataset_filtered)} examples")

if nemotron_dataset and len(nemotron_dataset) > 0:
    from datasets import concatenate_datasets
    
    # Make sure both have the same columns
    combined_dataset = concatenate_datasets([my_dataset_filtered, nemotron_dataset])
    print(f"   ✅ Combined dataset: {len(combined_dataset)} examples")
else:
    combined_dataset = my_dataset_filtered
    print(f"   ✅ Using HICRA only: {len(combined_dataset)} examples")

# === 4. Format for GRPO Training ===
print("📝 Formatting for GRPO...")
dataset_train = combined_dataset.map(format_prompt)

# Shuffle to mix the datasets
dataset_train = dataset_train.shuffle(seed=42)

# === 5. Load Test Set (HICRA only) ===
dataset_test = load_dataset(
    "json", 
    data_files="reasoning_dataset_v2_test.json", 
    split="train"
).map(format_prompt)

print(f"\n✅ Final Training Set: {len(dataset_train)} examples")
print(f"✅ Test Set: {len(dataset_test)} examples")
print(f"\nSample prompt format:")
print(dataset_train[0]['prompt'])

📂 Loading HICRA dataset...
   ✅ Loaded 729 HICRA examples
🌊 Streaming 3000 Nemotron math examples...


Resolving data files:   0%|          | 0/183 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/159 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/660 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/183 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/159 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/660 [00:00<?, ?it/s]

   Processed 500 examples, kept 498...
   Processed 1000 examples, kept 998...
   Processed 1500 examples, kept 1498...
   Processed 2000 examples, kept 1998...
   Processed 2500 examples, kept 2498...
   Processed 3000 examples, kept 2997...
   ✅ Loaded 2997 Nemotron examples (after length filter)
🔀 Combining datasets...
   HICRA after filter: 729 examples
   ✅ Combined dataset: 3726 examples
📝 Formatting for GRPO...

✅ Final Training Set: 3726 examples
✅ Test Set: 36 examples

Sample prompt format:
[{'content': 'You are aadvanced mathematical, logical reasoning assistant. Think through problems step by step.\nThe calculation is as important as the answer.\nRespond in the following format:\n<think>\n...\n</think>\n<answer>\n...\n</answer>', 'role': 'system'}, {'content': 'Evaluate the integral \\(\\int_0^{2\\pi} \\sqrt{\\sin^2(t) \\cos^2(t)} \\, dt\\).', 'role': 'user'}]


In [7]:
# Cell 5: Reward Functions
import re
from datetime import datetime
import json

# Strategic reasoning phrases (from HICRA paper)

PREVIOUS_GRAMS = [
    # Beginning a thought
    "let's analyze", "first we need", "to solve this", "let's assume", "i need to", "i'll go with",
    
    # Logic Connectors (The most important ones)
    "implies that", "consequently", "therefore",  "as we saw", "in any case", "i recall that",
     "i can compute it", "so it's fine", "so for any", "if i know", "so at point",
    "since", "given that", "conversely", "alternatively", 'but in a', "so, from above",
    "now is this a", "now, since the", "on the other hand", "we can actually",
    
    # Process Checks (Metacognition)
    "checking the", "verifying", "double check", "but wait", "identifying", "to make it", 
    "is only for", "it checks out", "i am considering", "find a pattern", "i'm given the", 
    "in first set", "must be a perfect", "made a mistake", "now back to", "that's not right",
    "notice that", "recall that", "we can conclude", "set the equation", "the same as", 
    "simpler as i did", "if i take a", "set up a", "is better if", "to be safe",  "i think that's it", "so yes, i can",
    "so no choice", "be related to", "in fact it's", "i'll do that", "to one side", "which it shouldn't be",
    "but it doesn't have", "looking back, it says",
    
    # Mathematical / Computational Actions
    "substituting", "calculating", "simplifying", "solving for", "derivative of", "now sum all", 
    "with a small", "for all real", "for the pair", "so ratio is", "is zero when", "of the form", "to compare the", "their product is"
    "now look at", "if i set", "as long as", "is symmetric in a", "is defined on",  "is positive", "is negative", 
    "hold for all", "is a function of", "so max is", "is a product of", "so min is", "as is standard",  
]

def extract_xml_answer(text: str) -> str:
    """Extract answer from <answer> tags."""
    if "<answer>" not in text:
        return text.strip()
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

# File to store incorrect answers for LLM-as-Judge evaluation
INCORRECT_ANSWERS_LOG = "incorrect_answers_for_judge_v2_phi4.jsonl"

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    Check if the model's answer matches the expected answer.
    Returns 2.0 for correct, 0.0 for incorrect.
    Logs incorrect answers to a file for LLM-as-Judge evaluation.
    """
    responses = [completion[0]['content'] for completion in completions]
    extracted = [extract_xml_answer(r) for r in responses]
    
    # Debug output (first item only)
    q = prompts[0][-1]['content'][:100]  # First 100 chars of question
    print(f"---\nQ: {q}...\nExpected: {answer[0]}\nExtracted: {extracted[0][:50]}...")
    
    rewards = []
    for i, (ext, ans, prompt, response) in enumerate(zip(extracted, answer, prompts, responses)):
        # Check if answer appears in extracted text
        if str(ans).strip() in ext:
            rewards.append(2.0)
        else:
            rewards.append(0.0)
            
            # Log incorrect answer for LLM-as-Judge
            log_entry = {
                "timestamp": datetime.now().isoformat(),
                "question": prompt[-1]['content'],  # User message
                "expected_answer": str(ans),
                "model_full_response": response,
                "model_extracted_answer": ext,
            }
            
            # Append to JSONL file
            with open(INCORRECT_ANSWERS_LOG, "a") as f:
                f.write(json.dumps(log_entry) + "\n")
    
    return rewards
print(f"✅ Reward function updated! Incorrect answers will be logged to '{INCORRECT_ANSWERS_LOG}'")

def reasoning_reward_func(completions, **kwargs) -> list[float]:
    """
    HICRA-inspired reward for reasoning structure.
    Gives bonus for using strategic reasoning phrases.
    """
    responses = [completion[0]['content'] for completion in completions]
    rewards = []
    
    for response in responses:
        score = 0.0
        response_lower = response.lower()
        
        # Check for strategic grams
        for gram in STRATEGIC_GRAMS:
            if gram in response_lower:
                score += 0.01
        
        # Bonus for using reasoning tags
        if "<think>" in response and "</think>" in response:
            score += 0.2
        if "<answer>" in response and "</answer>" in response:
            score += 0.1
        
        # Cap the reward
        rewards.append(min(score, 0.5))
    
    return rewards

def format_reward_func(completions, **kwargs) -> list[float]:
    """
    Reward for correct XML format AND stopping correctly.
    """
    rewards = []
    for completion in completions:
        response = completion[0]['content']
        
        # 1. Check if it has the tags
        has_tags = "<think>" in response and "</think>" in response and "<answer>" in response and "</answer>" in response
        
        # 2. Check if it rambles after the answer
        # We split by </answer> and check if there is significant text afterwards
        parts = response.split("</answer>")
        clean_stop = False
        if len(parts) > 1:
            # If the stuff after </answer> is just whitespace or EOS, it's good.
            # If it's another <think> block, it's bad.
            remainder = parts[1].strip()
            if len(remainder) < 5: # Tolerance for tiny noise
                clean_stop = True
        
        score = 0.0
        if has_tags:
            score += 0.5
        if clean_stop:
            score += 0.5 # Big bonus for stopping!
            
        rewards.append(score)
    return rewards

print("✅ Reward functions defined")

✅ Reward function updated! Incorrect answers will be logged to 'incorrect_answers_for_judge_v2_phi4.jsonl'
✅ Reward functions defined


### Chat Template (Save for Base models)

```
# Set Llama 3 chat template (required for GRPO with conversational data)
tokenizer.chat_template = """{% for message in messages %}{% if message['role'] == 'system' %}<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{{ message['content'] }}<|eot_id|>{% elif message['role'] == 'user' %}<|start_header_id|>user<|end_header_id|}
{{ message['content'] }}<|eot_id|>{% elif message['role'] == 'assistant' %}<|start_header_id|>assistant<|end_header_id|>
{{ message['content'] }}<|eot_id|>{% endif %}{% endfor %}{% if add_generation_prompt %}<|start_header_id|>assistant<|end_header_id|>
{% endif %}"""
print("✅ Chat template set!")
```

**Optional: Use More GPU**
You could also try:

- `num_generations=6` (more diverse rollouts per step)
- Or increase `max_seq_length=1280` in cell 3 if Nemotron answers are very long

### How to calculate the Math:

Base Model Cost: A 4-bit model takes ~0.7 GB per billion parameters.Phi-3.5 (3.8B) $\approx$ 2.5 GB.Phi-4 (14B) $\approx$ 10 GB.

Context Cost (The Killer): This is determined by num_generations $\times$ max_completion_length. 16 gens $\times$ 1536 tokens is a lot of data.

The Formula: If nvidia-smi (or equiv for whatever you use) says you are only using 12GB / 24GB, double your num_generations. 

This is the safest way to improve performance without changing the model architecture.

Parameter,Old Setting,New 

max_completion_length,2048,4096,Crucial. Allows for ~2-3 self-correction loops.

num_generations,20,8,"The Trade-off. 8 is the minimum for stable GRPO, but it frees up massive VRAM for the token length."

per_device_train_batch_size,1,1,Unsloth handles the rest.

gradient_accumulation_steps,1,4,"Since we lowered generations (effectively lowering batch size), we increase accumulation to keep the training stable."


In [ ]:
# Cell 8 (updated)
from trl import GRPOConfig, GRPOTrainer

# --- 2. Training Config for RTX 4090 ---
# Explicitly define these variables to avoid NameError
max_prompt_length = 512
max_completion_length = 8164  # I also tried 4096 tokens which seemed to work

training_args = GRPOConfig(
    output_dir="phi-4-hicra-reasoner",
    
    # OPTIMIZATION
    learning_rate=5e-6, # Keep low for stability
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    
    # MEMORY & BATCHING
    per_device_train_batch_size=1, # unsloth changes this saying:
    # Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
    # We will change the batch size of 1 to the `num_generations` of 20
    gradient_accumulation_steps=4, # Keep small when num_generations is high (16x)
    
    # GRPO SPECIFIC (The "Luxury" Settings)
    num_generations=8,       
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    
    # DURATION
    max_steps=1250, # Start small to test
    save_steps=100,
    logging_steps=1,
    
    # EFFICIENCY
    fp16=False,
    bf16=True, # 4090 loves Bfloat16
    report_to="tensorboard"
)

print(f"✅ Training configuration set")
print(f"   Prompt: {max_prompt_length} tokens, Completion: {max_completion_length} tokens")

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 20
✅ Training configuration set
   Prompt: 512 tokens, Completion: 2048 tokens


In [9]:
# Cell 9: Initialize Trainer
print("🚀 Initializing GRPO Trainer...")

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        correctness_reward_func,
        reasoning_reward_func,
        format_reward_func,
    ],
    args=training_args,
    train_dataset=dataset_train,
)

print("✅ Trainer initialized!")

🚀 Initializing GRPO Trainer...
✅ Trainer initialized!


Current settings `nvidia-smi` says: `6861MiB /  12282MiB`

In [10]:
# Cell 8: Run Training!
print("🏋️ Starting training...")
print("Note: First ~100 steps may show 0 reward. Be patient!")
print("="*50)

trainer_stats = trainer.train()

print("="*50)
print("✅ Training complete!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


🏋️ Starting training...
Note: First ~100 steps may show 0 reward. Be patient!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,726 | Num Epochs = 1 | Total steps = 1,250
O^O/ \_/ \    Batch size per device = 20 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (20 x 1 x 1) = 20
 "-____-"     Trainable parameters = 524,288,000 of 15,183,795,200 (3.45% trained)


Unsloth: Will smartly offload gradients to save VRAM!


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.28 GiB. GPU 0 has a total capacity of 23.48 GiB of which 1.30 GiB is free. Including non-PyTorch memory, this process has 21.22 GiB memory in use. Of the allocated memory 20.31 GiB is allocated by PyTorch, and 450.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Cell: Reinitialize for Training (run this before resuming after inference)
"""
import gc
import torch

# Force garbage collection
gc.collect()
torch.cuda.empty_cache()

# Reinitialize accelerator state
from accelerate.state import AcceleratorState
AcceleratorState._reset_state()
"""
# Then you MUST re-run the trainer initialization cell (Cell 9)
# before resuming training

Restart your kernel, run cells 1-9, then run your resume training cell. 🚀



In [ ]:
# Cell 11: Run Training!
print("🏋️ Resuming training from checkpoint...")

# Option A: Resume from the latest checkpoint automatically
# trainer.train(resume_from_checkpoint=True)

# Option B: Resume from a specific checkpoint (if you want to go back in time)
trainer.train(resume_from_checkpoint="./phi-3.5-hicra-reasoner/checkpoint-500")

print("="*50)
print("✅ Training complete!")

In [ ]:
# Cell 9: Save Model
import os
# Option 1: Save locally
output_path = "Phi-3_5-reasoning-unsloth-HICRA-v1"
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)
print(f"✅ Model saved to {output_path}")
# Option 2: Push to HuggingFace Hub (uncomment to use)
repo_name = "DataImaginations/Phi-4-reasoning-HICRA-v1"
hf_token = os.getenv('HF_TOKEN')

# Option 2: Push to HuggingFace Hub (uncomment to use)
# repo_name = "DataImaginations/Llama-1B-Reasoning-v1"
# hf_token = os.getenv('HF_TOKEN')
# 
# print(f"⏳ Pushing to {repo_name}...")
# model.push_to_hub_merged(
#     repo_name,
#     tokenizer,
#     save_method="merged_16bit",
#     token=hf_token
# )
# print("✅ Model pushed to Hub!")

In [ ]:
# Cell: Merge LoRA adapters and save for evaluation
from unsloth import FastLanguageModel

# Load the adapter model
model, tokenizer = FastLanguageModel.from_pretrained(
    "Phi-3_5-reasoning-unsloth-HICRA-v1",
    max_seq_length=2048,
    load_in_4bit=True,
)

# Merge and save in 16-bit
print("⏳ Merging adapters...")
model.save_pretrained_merged(
    "Phi-3_5-reasoning-HICRA-v1-merged",  # New path for merged model
    tokenizer,
    save_method="merged_16bit",  # Full precision merged weights
)
print("✅ Merged model saved!")


### This creates a clean, standard HuggingFace model without any Unsloth-specific patches.


In [ ]:
from unsloth import FastLanguageModel

# Load the model with adapter (NOT merged yet)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Phi-3_5-reasoning-unsloth-HICRA-v1",  # Your adapter folder
    max_seq_length=2048,
    load_in_4bit=True,
)

# Use Unsloth's save method - this properly handles the merge!
model.save_pretrained_merged(
    "Phi-3_5-HICRA-MERGED-16bit",
    tokenizer,
    save_method="merged_16bit",  # This saves a clean HF-compatible model
)

print("✅ Saved merged 16-bit model!")

In [ ]:
from unsloth import FastLanguageModel

# Load base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3.5-mini-instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

# Load your existing LoRA adapter directly
from peft import PeftModel
model = PeftModel.from_pretrained(model, "Phi-3_5-reasoning-unsloth-HICRA-v1")

# Merge adapter into base model
model = model.merge_and_unload()

# Save the merged model
model.save_pretrained("Phi-3_5-HICRA-MERGED")
tokenizer.save_pretrained("Phi-3_5-HICRA-MERGED")

print("✅ Merged model saved to Phi-3_5-HICRA-MERGED/")

## Test the Trained Model

In [ ]:
# Cell 11: Test Inference
from unsloth import FastLanguageModel

# Put model in inference mode
FastLanguageModel.for_inference(model)
# Create mask (1 for real tokens, 0 for padding) - mostly just all 1s for batch size 1
attention_mask = (inputs != tokenizer.pad_token_id).long()
# Test question
test_question = "A loan is repaid with 20 equal annual payments. The interest portion of the 16th payment is 400 and the interest portion of the 11th payment is 600. Find the interest portion of the 1st payment."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT.strip()},
    {"role": "user", "content": test_question}
]

# Tokenize with attention mask
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

# Generate 4 attempts at once
outputs = model.generate(
    input_ids=inputs,
    attention_mask=attention_mask,
    max_new_tokens=2048,
    temperature=0.7,   # Keep some randomness for diversity
    do_sample=True,
    num_return_sequences=4, # <--- KEY CHANGE: Ask for 4 opinions
    pad_token_id=tokenizer.eos_token_id,
    stop_strings=["</answer>", "<|end|>", "<|endoftext|>"],
    tokenizer=tokenizer,
)

# Print all 4 to see if the "Smart" version is hiding in there
print("--- Voting ---")
for i, out in enumerate(outputs):
    # Decode skipping the prompt
    response = tokenizer.decode(out[inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\nAttempt {i+1}:\n{response[:7048]}...") # Print start of answer
# Create attention mask (1 for all real tokens)
attention_mask = torch.ones_like(inputs)


# Actual Reasoning Solution from Claude

---

### Loan Amortization Problem
<think>
This is a loan amortization problem where we need to find the interest portion of the first payment given information about later payments.

Setting up the problem:


- Let A = the annual payment amount

- Let r = the interest rate per period

- The interest portion of payment k is: I_k = r × (balance before payment k)

Key insight: For loans with equal payments, the interest portions follow a pattern. Between consecutive payments:

I_{k+1} = I_k(1+r) - rA

This can be rearranged to show that (I_k - A) forms a geometric sequence:

I_k - A = (I_1 - A)(1+r)^{k-1}

Using the boundary condition: Since the loan is repaid in exactly 20 payments, after the 20th payment the balance is zero. This means:

I_{20} = Ar/(1+r)

Working through the algebra, we can derive:

I_k = A[1 - 1/(1+r)^{20-k+1}]

Applying known values:


- I_{11} = A[1 - 1/(1+r)^{10}] = 600

- I_{16} = A[1 - 1/(1+r)^{5}] = 400

Let x = (1+r)^5. Then:


- A(x² - 1)/x² = 600

- A(x - 1)/x = 400

From the second equation: A = 400x/(x-1)

Substituting into the first and simplifying:


- 400(x+1)/x = 600

- 400x + 400 = 600x

- x = 2

So (1+r)^5 = 2, and:

A = 400(2)/(2-1) = 800

Finally: I_1 = A[1 - 1/(1+r)^{20}] = 800[1 - 1/2^4] = 800(15/16) = 750

</think>

<answer>
The interest portion of the 1st payment is **750**.

To verify: With A = 800 and (1+r)^5 = 2:


- I_{11} = 800[1 - 1/4] = 600 ✓

- I_{16} = 800[1 - 1/2] = 400 ✓


</answer>

# Bench for unsloth_HICRA

In [ ]:
# benchmark cell
import lm_eval
from lm_eval.models.huggingface import HFLM

# 1. Load your local model
# If you just saved your model to "llama-1b-reasoning-final", point to that folder.
print("⏳ Loading model for evaluation...")

# We wrap the model in the Harness's HFLM wrapper
# 'pretrained' can be a local path OR a Hub ID (e.g., "david-barnes/my-model")
llm = HFLM(
    pretrained="Phi-3_5-HICRA-MERGED-16bit",  # Your merged model
    tokenizer="unsloth/Phi-3.5-mini-instruct-bnb-4bit",  # Add this line!
    batch_size=1,
    trust_remote_code=True,
    dtype="bfloat16"
)

# 2. Define the tasks you want
# These key names correspond to the harness registry.
# Note: "minerva_math" is often split by subject (algebra, etc), 
# so we usually run the main "math" group or specific subtasks.
task_list = [
    "arc_challenge",
    "hellaswag", 
    "winogrande",
    "piqa",
    "mmlu",
    "gsm8k",
    "truthfulqa_mc2",
]

print(f"🚀 Running BASELINE evaluation on: {task_list}...")
hicra_results = lm_eval.simple_evaluate(
    model=llm,
    tasks=task_list,
    num_fewshot=0,
    limit=100,  # Same limit for comparison
    log_samples=True,
)
# 4. Print a Pretty Table
from lm_eval.utils import make_table
print(make_table(hicra_results))

# 5. Save detailed results to JSON (Crucial for your blog!)
import json
with open("phi_3.5_hicra_reasoner_v1_benchmark_results.json", "w") as f:
    json.dump(hicra_results, f, indent=2)

# Soft VRAM clear

In [ ]:
import torch
import gc

# 1. Delete the Python variables holding the model
# (Wrap in try/except so it doesn't crash if they are already gone)
try:
    del model
    del tokenizer
    del trainer
except NameError:
    print("Variables already deleted or not defined.")

# 2. Python Garbage Collection (Clears CPU RAM)
gc.collect()

# 3. PyTorch Cache Clearing (The most important step for VRAM)
torch.cuda.empty_cache()

# Verify: Print current memory usage
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU Memory Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")